In [ ]:
import os
nnn = 1
os.environ["OMP_NUM_THREADS"] = str(nnn) # export OMP_NUM_THREADS=1
os.environ["OPENBLAS_NUM_THREADS"] = str(nnn) # export OPENBLAS_NUM_THREADS=1
os.environ["MKL_NUM_THREADS"] = str(nnn) # export MKL_NUM_THREADS=1
os.environ["VECLIB_MAXIMUM_THREADS"] = str(nnn) # export VECLIB_MAXIMUM_THREADS=1
os.environ["NUMEXPR_NUM_THREADS"] = str(nnn)  # export NUMEXPR_NUM_THREADS=1

os.environ["TOKENIZERS_PARALLELISM"] = "false"
import pandas as pd
from pathlib import Path

import shutil
USE_SLURM = False
if shutil.which("squeue"):
    print("Slurm is available on this system.")
    USE_SLURM = True
else:
    print("Slurm is not available.")

In [ ]:
from TELF.pipeline.blocks import DataBundle, SAVE_DIR_BUNDLE_KEY, SOURCE_DIR_BUNDLE_KEY
from TELF.pipeline import BlockManager  

from TELF.pipeline.blocks import (
    DataBundle,
    VultureCleanBlock,
    BeaverVocabBlock,
    OrcaBlock,
    WolfBlock,
    CleanDuplicatesBlock,
    CleanAffiliationsBlock,
    BeaverDocWordBlock,
    SemanticHNMFkBlock,
    ArticFoxBlock,
    TermAttributionBlock,
    LoadTermsBlock,
    TermAttributionBlock,
    SBatchBlock,
    ClusteringAnalyzerBlock,
    PeacockStatsBlock,
    PipelineSummaryBlock,
    CollectHNMFkLeafBlock,
    SpacyNERBlock,
    TermiteNeo4jBlock,
    TermiteVectorBlock,
    AffiliationsAndAuthorsBlock,
    TermTableBlock,
)

# Load Data

In [ ]:
df = pd.read_csv(Path("..") / ".." / ".." /"data" / "sample2.csv").head(50)
EXAMPLE_OUTPUT = Path( "example_results") / 'semantic_HNMFk_collection_slurm_option' 
bundle = DataBundle({'Default.df':df, 
                     SAVE_DIR_BUNDLE_KEY: EXAMPLE_OUTPUT,
                     SOURCE_DIR_BUNDLE_KEY: EXAMPLE_OUTPUT})
df.info()


# Build the Blocks

In [ ]:
duplicate_cleaner_block = CleanDuplicatesBlock()
orca_block = OrcaBlock()
clean_affiliations_block = CleanAffiliationsBlock()
vulture_block = VultureCleanBlock(verbose=True, 
                                  use_substitutions=True,
                                  init_settings={"n_jobs":-1, 'parallel_backend': 'threading'})
vocab_block = BeaverVocabBlock(call_settings={'min_df':3, 'max_df':0.6, 'max_features':10000})

In [ ]:
terms_block = LoadTermsBlock( call_settings={SOURCE_DIR_BUNDLE_KEY: Path('../../../data/sample_terms3.md')})
out_terms = terms_block(bundle=bundle)
out_terms.substitutions


In [ ]:
matrix_block = BeaverDocWordBlock(tag="DocWord", needs=("df", "vocabulary",))

In [ ]:
nmfk_params = {
            "n_perturbs": 2,
            "n_iters": 2,
            "epsilon": 0.015,
            "n_jobs": -1,
            "init": "nnsvd",
            "use_gpu": True,
            "save_output": True,
            "collect_output": True,
            "predict_k_method": "sill",
            "verbose": True,
            "nmf_verbose": False,
            "transpose": False,
            "sill_thresh": 0.8,
            "pruned": True,
            "nmf_method": "nmf_fro_mu",
            "calculate_error": True,
            "predict_k": True,
            "use_consensus_stopping": 0,
            "calculate_pac": True,
            "consensus_mat": True,
            "perturb_type": "uniform",
            "perturb_multiprocessing": False,
            "perturb_verbose": False,
            "simple_plot": True,
            "k_search_method": "bst_pre",
            "H_sill_thresh": 0.1,
            "clustering_method": "kmeans",
            "device": -1,
        }

In [ ]:
semantic_hfactor_block = SemanticHNMFkBlock(
    needs=("DocWord.X", "df", "vocabulary", ),
    init_settings={
        "depth":2, 
        "sample_thresh":5,
        "Ks_deep_max":30,
        "nmfk_params":[nmfk_params],
    },
    call_settings={
        "Ks":range(2, 10),
    }
)

if USE_SLURM:
    sbatch_hnmfk = SBatchBlock(
        wrapped_block=semantic_hfactor_block,
        venv_type="conda",
        venv_path="TELF2",
    )

    hnmfk_block = sbatch_hnmfk
else:
    hnmfk_block = semantic_hfactor_block



In [ ]:
wolf_coauthor_block = WolfBlock(tag="WolfAuthor", category='co-author')
wolf_coaffiliation_block = WolfBlock(tag="WolfAffil", category='co-affiliation')
term_attribute_block =   TermAttributionBlock( )
post_process_block = ArticFoxBlock(call_settings={"ollama_model":"llama3.2:3b-instruct-fp16",
                                                  "steps": ["post",
                                                            # "label",
                                                            "stats"]
                                                  })

In [ ]:
hnmfk_analyzer = ClusteringAnalyzerBlock(
    tag='HNMFAnalyzer',
    mode='hnmf'
)

summary_block = PipelineSummaryBlock()
# peacock_block = PeacockStatsBlock()
peacock_block=PeacockStatsBlock(mode="hnmfk",    load_checkpoint=False,   # don't load
    checkpoint=False,        # don't save
    skip_completed=False     # render even if PeacockStats.done exists
    )

HNMFK_OUTPUT_PATH = "./example_results/semantic_HNMFk_collection_slurm_option/07_SemanticHNMFk"
collect_leaves = CollectHNMFkLeafBlock(  call_settings={"hnmfk_dir": HNMFK_OUTPUT_PATH},)

affiliation_author_block = AffiliationsAndAuthorsBlock(
    call_settings={
        "min_total_papers": 20,
        "country_filter": None,            # or "United States" / "unknown"
        "partition_by_year": True,
        "per_year_output_dir": None,       # defaults to <tag>/by_year when True
        "countries": ["United States"],    # for top authors (optional)
        "top_n": 10,
    }
)

In [ ]:
spacyNERblock = SpacyNERBlock()

TermiteNeo4jBlock(call_settings={"settings_yaml_path": "/path/to/settings.yml"})


neo4j_block = TermiteNeo4jBlock(
    needs=("spaceyNER.df", "leaf_labels_csv"),
    call_settings={
    "settings_yaml_path": "termite.yml",
    "raw_csv_path": bundle.get("LeafDataLabels.leaf_data_csv"),
    "neo4j_uri": "neo4j://localhost:7666",
    "neo4j_user": "neo4j",
    "neo4j_pass": "local_password",
})

vector_store_block = TermiteVectorBlock(call_settings={
    "raw_csv_path": bundle.get("LeafDataLabels.leaf_data_csv"),
    "id_column": "eid",
    "text_column": "abstract",
    "index_name": "termite_vectors_test_e2e",
    "model_name": "malteos/scincl",
    "env": {  # optional override; defaults match your script
        "EMBEDDING_STORE": "opensearch",
        "OS_HOST": "localhost",
        "OS_PORT": "9200",
        "OS_USE_SSL": "false",
    },
    # Optional smoke test:
    # "test_query_text": "What problem in real-world malware labeling does the HNMFk Classifier aim to solve?",
    # "test_k": 5,
})

# Block Manager

In [ ]:
manager = BlockManager(
    blocks = [
        duplicate_cleaner_block,
        vulture_block,
        orca_block,
        clean_affiliations_block,
        vocab_block,
        matrix_block,
        hnmfk_block,
        # term_attribute_block,
        wolf_coauthor_block,
        wolf_coaffiliation_block,
        post_process_block,
        peacock_block,
        collect_leaves,
        summary_block,
        spacyNERblock,
        affiliation_author_block,
        # neo4j_block,
        # vector_store_block
    ],
    databundle=bundle,  
    progress   = True,          # see which block is executing
    capture_output='file',
)

In [ ]:
bundle = manager()

In [ ]:
bundle.keys()


In [ ]:
from TELF.pipeline import KernelTiedServer
srv = KernelTiedServer(
    [
        "zsh", 
        "../../Lynx/start_lynx.sh", 
        "-p",
        HNMFK_OUTPUT_PATH
    ], 
    log_dir=".logs/lynx"
).start()

import time; time.sleep(5)  # wait 5s
print(srv.status()); srv.tail(100)          # view last 100 lines of stdout
# srv.stop()     

In [ ]:
srv.tail(10000)    